In [0]:
pip install -q databricks-sdk[openai]

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
import time
import requests, json, re

In [0]:
CANDIDATE_PROFILE = """"Data Engineer with 4+ years of experience designing and optimizing scalable ETL pipelines, data warehouses, and cloud-based solutions. Skilled in Azure, PySpark, and SQL with a proven record of improving cost efficiency, reliability, and performance. Seeking to leverage expertise in Azure Data Engineering and modern cloud platforms to build robust, data-driven solutions that support business growth.",
      "Technical Skills": 
      {
          "Programming Language": "Python, SQL, PySpark",
          "Cloud & Data Services": "Azure Databricks, Azure Data Factory (ADF), ADLS, Azure Synapse Analytics (ASA)",
          "Databases": "SQL Server, Oracle, Snowflake",
          "Cloud Platforms": "Azure",
          "Tools": "GitHub, Azure DevOps CI/CD, Prefect",
          "Concepts": "ETL/ELT, Medallion Architecture, Data Modeling, Data Warehousing, API Integration",
          "AI Technologies": "langchain, vector db, RAG, MCP, Databricks with AI"
      }"""

In [0]:
def get_matching_score(description):
  w = WorkspaceClient()
  client = w.serving_endpoints.get_open_ai_client()



  prompt = f"""You are an expert technical recruiter and ATS evaluator.
  Task:
  Compare the provided Job Description (JD) against the Candidate Profile and calculate a match score between 0 and 100.
  Job Description:
  {description}
  
  Candidate Profile:
  {CANDIDATE_PROFILE}"""

  system_msg = (
          "You are an expert technical recruiter and ATS evaluator. "
          """
          CRITICAL OUTPUT REQUIREMENTS:
          - Return ONLY a valid JSON object.
          - Do NOT include markdown.
          - Do NOT wrap JSON in ```json blocks.
          - Do NOT provide explanations outside the JSON.
          - Do NOT include introductory text such as "Here is the result".
          - The response must start with { and end with }.
          - If any field is unknown, use null.
          - Output must be parseable by json.loads() without modification.

          Mandatory Eligibility Rule:
          1. Extract the minimum years of experience required from the JD.
          2. Candidate experience = {candidate_experience_years}.
          3. If the JD explicitly requires more than 5 years of experience (minimum experience > 5), return:
          - score = 0
          - eligible = false
          - reason = "Candidate has less than the required minimum experience."
          4. Do not continue skill matching if Rule #3 is triggered.
          
          Scoring Rules (only if eligible):
          1. Compare candidate skills, experience, tools, technologies, and domain knowledge against the JD.
          2. Assign scores:
          - Required skills match: 70%
          - Preferred/Nice-to-have skills match: 20%
          - Relevant domain/project experience: 10%
          3. Calculate a final score between 0 and 100.
          4. Be strict. Do not award points for unrelated skills."""
          """
          Output Format (JSON only) in all conditions:
  
          {
          "eligible": true,
          "score": 85,
          "experience_required": "3+ years",
          "candidate_experience": "5 years",
          "matched_skills": [
              "Azure Data Factory",
              "PySpark",
              "SQL"
          ],
          "missing_skills": [
              "Terraform",
              "Kafka"
          ],
          "reason": "Strong match on required Azure data engineering skills with minor gaps in preferred technologies."
          }
          """
      )


  resp = client.chat.completions.create(
          model="databricks-llama-4-maverick",
          messages=[
              {"role": "system", "content": system_msg},
              {"role": "user", "content": prompt},
          ],
          temperature=0.1,
          max_tokens=300,
      )
  return(resp.choices[0].message.content)

In [0]:
def safe_json_parse(text):
    try:
        data = json.loads(text)
        return data
    except json.JSONDecodeError:
        # Extract first JSON object using regex
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise


In [0]:
def score_job(jobs, url_col, desc_col, skill_col):
    scored_jobs = []
    for job in jobs:
        description = job[desc_col]
        if job[skill_col]:
            description += ", " + job[skill_col]
        time.sleep(1)
        score = safe_json_parse(get_matching_score(description))["score"]
        data = {
            "url": job[url_col],
            "score": int(score)
        }
        scored_jobs.append(data)
    if not scored_jobs: scored_jobs = [{"url": "", "score": 0}]
    return scored_jobs

In [0]:
def score_job_workday(jobs, url_col):
    scored_jobs = []
    for job in jobs.collect():
        try:
            api_url = job["api_url"]
            response = requests.get(api_url)
            description = response.json()["jobPostingInfo"]["jobDescription"]
            time.sleep(1)
            score = safe_json_parse(get_matching_score(description))["score"]
            data = {
                "url": job[url_col],
                "score": int(score)
            }
            scored_jobs.append(data)
        except:
            pass
    if not scored_jobs: scored_jobs = [{"url": "", "score": 0}]
    return scored_jobs

In [0]:
def send_message(message, chat_ids):
    TOKEN = dbutils.secrets.get("job_notification", "Telegram_Token")
    CHAT_ID = chat_ids
    MESSAGE = message

    url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
    for user in CHAT_ID:
        payload = {
            "chat_id": user,
            "text": MESSAGE,
            "parse_mode": "HTML"
        }

        res = requests.post(url, data=payload)

In [0]:
def form_message_and_send(combined_df, chat_ids):
    s = ''
    count = 1
    tc = 1
    tl = combined_df.count()
    for row in combined_df.collect():
        # print(row)
        s += f"""{tc}. <a href = "{row["url"]}" >{row["title"]}</a>\n<b>{row["company"]}</b>, <i>{row["location"]}, {row["hrs_ago"]}</i> \n\n"""
        if count==20 or tc==tl:
            send_message(s, chat_ids)
            count = 0
            s = ''
        count+=1
        tc+=1